In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from frigeo import racine_projet
from frigeo.perimetre import decouvrir

# PARAMETRE A MODIFIER CHAQUE MOIS
# Dernier mois déposé dans data/raw, au format AAAA-MM
PERIODE_FIN = "2026-03"

In [ ]:
DATA = racine_projet() / "data"
perimetre = decouvrir(DATA, PERIODE_FIN)
perimetre.verifier()
print(perimetre.resume())
perimetre.tableau()

In [ ]:
import csv
import io
from collections import Counter


def format_declare(nom):
    """Renvoie (flux, encodage, séparateur) selon le dossier de données."""
    if nom.startswith("pieces_interventions"):
        return "pieces_interventions", "utf-8", ";"
    if nom.startswith("interventions"):
        return "interventions", "utf-8", ";"
    if nom.startswith(("FEC_extraction", "999000001FEC")):
        return "fec", "iso-8859-15", "\t"
    if nom == "clients.csv":
        return "clients", "cp1252", ";"
    if nom == "fournisseurs.csv":
        return "fournisseurs", "utf-8", ";"
    return None


def sonder_texte(chemin, encodage, sep):
    brut = (DATA / chemin).read_bytes()
    res = {
        "bom_utf8": brut.startswith(b"\xef\xbb\xbf"),
        "crlf": brut.count(b"\r\n"),
        "lf_seul": brut.count(b"\n") - brut.count(b"\r\n"),
        "decodage": "ok",
    }
    try:
        texte = brut.decode(encodage)
    except UnicodeDecodeError as e:
        res["decodage"] = f"echec a l'octet {e.start}"
        return res, None
    lecteur = csv.reader(io.StringIO(texte, newline=""), delimiter=sep)
    entete = next(lecteur, [])
    longueurs = Counter(len(ligne) for ligne in lecteur)
    res.update(
        colonnes_entete=len(entete),
        enregistrements=sum(longueurs.values()),
        lignes_physiques=lecteur.line_num,
        longueurs=dict(longueurs.most_common(4)),
    )
    return res, tuple(entete)


lignes, entetes = [], {}
for f in perimetre.fichiers:
    fmt = format_declare(f["fichier"])
    if fmt is None:
        continue
    flux, encodage, sep = fmt
    res, entete = sonder_texte(f["chemin"], encodage, sep)
    lignes.append({"fichier": f["chemin_relatif"], "flux": flux, **res})
    entetes.setdefault((flux, entete), []).append(f["chemin_relatif"])

structure_texte = pd.DataFrame(lignes)
display(structure_texte)

print("En-têtes distincts par flux")
for (flux, entete), fichiers in entetes.items():
    print(flux, len(fichiers), "fichier(s)", "|".join(entete or ()))

In [ ]:
import openpyxl

lignes, entetes = [], {}
for f in perimetre.fichiers:
    if not f["fichier"].endswith(".xlsx"):
        continue
    classeur = openpyxl.load_workbook(f["chemin"], read_only=True, data_only=True)
    for feuille in classeur.worksheets:
        entete = next(feuille.iter_rows(min_row=1, max_row=1, values_only=True), ())
        lignes.append(
            {
                "fichier": f["chemin_relatif"],
                "feuille": feuille.title,
                "lignes": feuille.max_row,
                "colonnes": feuille.max_column,
            }
        )
        entetes.setdefault((feuille.title, tuple(entete)), []).append(f["chemin_relatif"])
    classeur.close()

structure_excel = pd.DataFrame(lignes)
display(structure_excel)

print("En-têtes distincts par feuille")
for (feuille, entete), fichiers in entetes.items():
    print(feuille, len(fichiers), "fichier(s)", "|".join(str(c) for c in entete))

In [ ]:
from frigeo.chargement import bilan_chargement, charger_tout

tables = charger_tout(perimetre)
bilan = bilan_chargement(tables)

print({table: donnees.shape for table, donnees in tables.items()})
bilan.groupby("table")["lignes_chargees"].sum()

In [ ]:
from pathlib import Path

attendu_texte = structure_texte.assign(
    fichier=structure_texte["fichier"].map(lambda p: Path(p).name)
).set_index("fichier")["enregistrements"]

attendu_excel = structure_excel.assign(
    fichier=structure_excel["fichier"].map(lambda p: Path(p).name),
    lignes_donnees=structure_excel["lignes"] - 1,
).set_index(["fichier", "feuille"])["lignes_donnees"]

comparaison = bilan.copy()
comparaison["attendu"] = [
    attendu_excel.get((f, s)) if isinstance(s, str) else attendu_texte.get(f)
    for f, s in zip(comparaison["fichier_source"], comparaison["feuille_source"])
]
comparaison["ecart"] = comparaison["lignes_chargees"] - comparaison["attendu"]
comparaison[comparaison["ecart"] != 0]

In [ ]:
suspects = []
for table, donnees in tables.items():
    for colonne in donnees.columns:
        if donnees[colonne].dtype != object:
            continue
        nombre = donnees[colonne].astype(str).str.contains("Ã|Â|\ufffd", regex=True).sum()
        if nombre:
            suspects.append({"table": table, "colonne": colonne, "cellules": int(nombre)})

print("Colonnes avec séquences suspectes", len(suspects))
pd.DataFrame(suspects)

In [ ]:
motif = "Ã|Â|\ufffd"
for table, colonne in [("interventions", "client"), ("fec", "CompAuxLib"), ("clients", "raison_sociale")]:
    donnees = tables[table]
    suspects = donnees[donnees[colonne].astype(str).str.contains(motif, regex=True)]
    display(suspects[["fichier_source", "num_ligne_source", colonne]])

In [ ]:
from pathlib import Path

import pandas as pd

from frigeo import racine_projet

motif = "Ã|Â|\ufffd"
cibles = [("interventions", "client"), ("fec", "CompAuxLib"), ("clients", "raison_sociale")]
lecture = {"interventions": "utf-8", "fec": "iso8859_15", "clients": "cp1252"}
reparation = {"interventions": "cp1252", "fec": "iso8859_15", "clients": "cp1252"}


def ligne_brute(nom_fichier, numero):
    chemin = next((Path(racine_projet()) / "data").rglob(nom_fichier))
    with open(chemin, "rb") as f:
        for i, brut in enumerate(f, start=1):
            if i == numero:
                return brut


def reparer(valeur, encodage):
    try:
        return valeur.encode(encodage).decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return None


lignes = []
for table, colonne in cibles:
    donnees = tables[table]
    masque = donnees[colonne].fillna("").astype(str).str.contains(motif, regex=True)
    ailleurs = set(donnees.loc[~masque, colonne].astype(str))
    for _, ligne in donnees[masque].iterrows():
        valeur = ligne[colonne]
        octets = valeur.encode(lecture[table])
        brut = ligne_brute(ligne["fichier_source"], ligne["num_ligne_source"])
        reparee = reparer(valeur, reparation[table])
        lignes.append({
            "table": table,
            "fichier_source": ligne["fichier_source"],
            "num_ligne_source": ligne["num_ligne_source"],
            "valeur_lue": valeur,
            "octets": octets,
            "dans_la_ligne_brute": octets in brut,
            "forme_reparee": reparee,
            "reparee_presente_ailleurs": reparee in ailleurs,
        })
display(pd.DataFrame(lignes))

### Cellules à l'encodage suspect

21 cellules portent une séquence typique d'un mauvais encodage : 6 dans `interventions.client`, 7 dans `fec.CompAuxLib` et 8 dans `clients.raison_sociale`.

Les octets des lignes sources montrent deux mécanismes. Dans les interventions, le texte est doublement encodé (UTF-8 relu en Windows-1252, puis réencodé en UTF-8). Dans le FEC et les clients, des octets UTF-8 bruts sont présents dans un fichier déclaré en ISO-8859-15 ou en Windows-1252. Dans chaque cas, la forme réparée est cohérente, et elle existe ailleurs dans la colonne pour les interventions et le FEC.

Ces cellules restent telles quelles dans les données chargées. Elles rejoindront la table des anomalies (étape 5) et feront l'objet d'une règle de nettoyage traçable (étape 6).